# 01 - ICU Cohort Extraction

Short summary:
[01 - ICU Cohort Extraction (SHORT)](../docs/01_icu_cohort_SHORT.md)

----

## 1. Objective
Build the base ICU cohort from MIMIC-IV using BigQuery and verify the integrity of the extracted ICU-stay-level tables.

The current SQL selects adults using estimated admission age `anchor_age + admission_year - anchor_year >= 18` and keeps the first ICU stay within each hospital admission. A patient can contribute repeated admissions. Age remains approximate; `anchor_age=91` is an aggregated oldest-age group, not an exact recovered age. See [the age and outcome definitions](../docs/METHODS_OVERVIEW.md).

----

## 2. Data Processing Steps


### 2.1 Common BigQuery Utilities


In [1]:
# Shared ADC client with US location, dry runs and per-job byte guards.
from pathlib import Path
import sys
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "scripts"))
from notebook_bq_runtime import client_for_notebook, run_sql_script, query_to_df
client = client_for_notebook()
print("Connected to BigQuery project: mimic-iv-portfolio (US)")

Connected to BigQuery project: mimic-iv-portfolio (US)


### 2.2 Build Cohort Tables in BigQuery


In [2]:
# Excecute SQL script (01_extract_base.sql)
run_sql_script("../sql/01_extract_base.sql")

Executed SQL script: 01_extract_base.sql


### 2.3 Sanity Checks


In [3]:
# Sanity checks for the created base ICU cohort table
# Row-level data are loaded for schema validation only; no rows are displayed.
df_cohort_icu = query_to_df("""
    SELECT *
    FROM `mimic-iv-portfolio.copd_raas.cohort_icu`
""")

df_cohort_icu.info()
print("Shape:", df_cohort_icu.shape)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 85242 entries, 0 to 85241
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   subject_id           85242 non-null  Int64         
 1   hadm_id              85242 non-null  Int64         
 2   stay_id              85242 non-null  Int64         
 3   intime               85242 non-null  datetime64[us]
 4   outtime              85228 non-null  datetime64[us]
 5   icu_los              85228 non-null  float64       
 6   age                  85242 non-null  Int64         
 7   anchor_age           85242 non-null  Int64         
 8   age_anchor_topcoded  85242 non-null  Int64         
 9   gender               85242 non-null  object        
 10  anchor_year          85242 non-null  Int64         
 11  anchor_year_group    85242 non-null  object        
dtypes: Int64(7), datetime64[us](2), float64(1), object(2)
memory usage: 8.4+ MB
Shape: (8524

---

## 3. Output Dataset

**Input Dataset**
- MIMIC-IV ICU stay and patient tables in BigQuery.

**Output Dataset**
- `mimic-iv-portfolio.copd_raas.cohort_icu`

**Downstream Usage**
- This ICU-stay-level table is the base cohort input for COPD cohort construction in `02_cohort_and_exposures.ipynb`.

---

## Next Step

Proceed to:
- [02 - Cohort and Exposure Construction](02_cohort_and_exposures.ipynb)

This next notebook defines the COPD ICU cohort and pre-ICU RAAS exposure indicators.
